# 1 · LangChain — The Foundation Layer

**What you'll learn**
1. **LCEL** — compose a prompt, model, and parser into a chain with the `|` pipe
2. **Streaming** — get the answer token-by-token with the *same* chain
3. **Tool calling** — how an LLM decides to use a function (the primitive every agent is built on)

> ▶️ Run the cells **top to bottom** with **Shift + Enter**.

## 0 · Setup

In [ ]:
# Make the repo root importable so `import config` works from the notebooks/ folder
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from config import get_langchain_llm, assert_key
assert_key()                 # fails early if your API key isn't set
llm = get_langchain_llm()    # one chat model, pointed at your gateway
print("Gateway ready.")

## 1 · LCEL — compose with the pipe `|`

**LCEL** (LangChain Expression Language) lets you wire components together with `|`,
read left→right like a Unix pipe:

`prompt  |  model  |  parser`

- **prompt** — fills your variables into a template
- **model** — sends it to the LLM
- **parser** — pulls the plain text out of the model's reply

> **Key idea:** every piece is a *Runnable* with the same interface, so you can
> compose them freely and get `stream` / `batch` / `async` for free.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise assistant for AI engineers."),
    ("human", "Explain {topic} in exactly two sentences."),
])

chain = prompt | llm | StrOutputParser()          # <-- this is LCEL

print(chain.invoke({"topic": "the difference between an LLM and an agent"}))

### 1.1 · Same chain, now streaming

Swap `.invoke()` for `.stream()` and print each chunk as it arrives — same chain,
better UX (the answer "types out" instead of appearing all at once).

In [ ]:
for chunk in chain.stream({"topic": "why observability matters for agents"}):
    print(chunk, end="", flush=True)   # end="" keeps it on one line; flush shows it live

## 2 · Tool calling — how an LLM takes action

An LLM only produces text — it can't run code. **Tool calling** lets the model
*ask* to run one of your functions.

- **`@tool`** turns a Python function into a tool. Its **type hints** become the
  argument schema and its **docstring** tells the model *when* to use it.
- **`bind_tools([...])`** attaches those tools to the model.

In [ ]:
from langchain_core.tools import tool

@tool
def get_stock_level(sku: str) -> int:
    'Return the number of units in stock for a given product SKU.'   # <- the model reads this
    return {"scout": 12, "hauler": 3, "sentinel": 0}.get(sku.lower(), 0)

llm_with_tools = llm.bind_tools([get_stock_level])
msg = llm_with_tools.invoke("How many Hauler units are in stock?")

print(msg.tool_calls)   # the model's REQUEST to call the tool

### 2.1 · The catch: the model *requests*, you *execute*

Notice the output above is a **request** (`{'name': 'get_stock_level', 'args': {'sku': 'Hauler'}}`)
— the number `3` is **not** there. The model can't run code; it only asks. **You**
run the function and feed the result back so the model can finish the answer:

In [ ]:
from langchain_core.messages import HumanMessage

q = "How many Hauler units are in stock?"
ai_msg = llm_with_tools.invoke(q)

# 1) run each tool the model asked for -> each returns a ToolMessage (matched by id)
messages = [HumanMessage(q), ai_msg]
for tc in ai_msg.tool_calls:
    messages.append(get_stock_level.invoke(tc))

# 2) send the results back -> model writes the final natural-language answer
final = llm_with_tools.invoke(messages)
print(final.content)

## 2.2 · Structured output

Need guaranteed JSON, not prose? `with_structured_output(Schema)` forces the model to
return an object matching a **Pydantic** model — validated for you (it's tool calling
under the hood). Perfect for extraction / classification.

In [ ]:
from pydantic import BaseModel, Field

class Product(BaseModel):
    name: str = Field(description="the product name")
    units_in_stock: int = Field(description="number of units in stock")

structured = llm.with_structured_output(Product)
result = structured.invoke("The Acme Hauler currently has 3 units in stock.")
print(result)                 # a validated Product object
print(result.units_in_stock)  # -> 3

## 3 · Recap

- **LCEL** (`prompt | model | parser`) composes components; `.invoke` / `.stream` for free.
- **Tool calling** = the model *requests* a function; *you* execute it and feed the result back.
- That request → execute → feed-back **loop** is manual here...

➡️ **Next (notebook 2): LangGraph runs that loop for you** — automatically, with memory and control flow.

## 🧪 Your turn
1. Add a second tool `get_price(sku)` that returns a price per SKU.
2. Bind **both** tools and ask a question that needs both, e.g. *"What's the total value of our Hauler stock?"*
3. Print `.tool_calls` — did the model request **both** tools in one turn (parallel tool calling)?
4. (Stretch) Close the loop: run both tools and get the final sentence.

In [ ]:
# your code here